# 07 — RAG agéntico: el juez selecciona los chunks (vía Ollama)

Segundo diseño de RAG agéntico, el que pidió el usuario: el sistema SOLO califica los chunks del RAG híbrido y selecciona los útiles con un juez. Sin graph_query, sin calculator.

Por pregunta: (1) retrieval híbrido top-10 (BM25 Okapi + MiniLM, RRF k=60); (2) UN llamado al juez con los 10 chunks numerados, que devuelve un veredicto por chunk (UTIL / NO UTIL) con su razón corta — máximo 3 reintentos si los veredictos no se pueden parsear; si a los 3 sigue sin veredictos válidos, la pregunta se abstiene sin llamar al lector; (3) el lector responde ÚNICAMENTE con los chunks seleccionados (mismo lector, mismo condense, mismo prompt del RAG baseline); si el juez no selecciona nada, abstención honesta.

**Juez y lector vía ChatOllama** contra el servidor local (`JUDGE_OLLAMA_MODEL`, default `qwen2.5:7b`; `qwen2.5:1.5b` para paridad con el run en CPU). La lógica de retrieval, del juez/lector y del scorer va inline en este notebook, sin correr ni importar scripts del repo (procedencia: `rag_baseline/retrieve_hybrid.py`, `agentic_rag/judge_rerank.py` y `rag_baseline/answer.py` para el juez y el lector, `agentic_rag/score_judge.py` para el scorer).

**El intento anterior queda como negativo documentado:** el agente ReAct con LangChain + LangGraph y 3 herramientas sacó 2/12 en CPU y nunca usó `vector_search` ni `calculator` (24/24 llamadas fueron `graph_query`): midió un agente solo-grafo, no un RAG agéntico. Ese run no se toca (`agentic_rag/results_agentic.md`).

**Prerrequisitos:** notebook 01 (chunks). El retrieval híbrido top-10 se genera en este mismo notebook.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
JUDGE_OLLAMA_MODEL = "qwen2.5:7b"   # juez y lector: 7b (default, el que muestra lo agentico)
                                     # o qwen2.5:1.5b para paridad con el run en CPU
K = 10
RETRIEVAL_K10 = REPO + "/rag_baseline/retrieval_hybrid_k10.json"
TRACE_OUT = REPO + "/agentic_rag/run_trace_judge_colab.jsonl"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive.
3. **Solo si no lo subes**, generarlo aquí mismo con la lógica de extracción inline (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal. La lógica es la de `pipeline/extract.py` del repo, copiada en la celda: el notebook no ejecuta scripts del repo.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive.")
    print("Si no lo subes, el notebook lo genera aquí mismo con la lógica inline (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar los chunks aquí mismo...")
        print("(lógica de extracción inline: descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        # El PDF está cifrado con AES (contraseña vacía): pypdf necesita el
        # paquete opcional `cryptography` para leerlo, que `pip install pypdf`
        # no trae. Se verifica aquí para que el último recurso funcione.
        try:
            import pypdf
            import cryptography
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1", "cryptography>=3.1"], check=True)
            import pypdf
            import cryptography
        import urllib.request
        from pypdf import PdfReader

        # Lógica de extracción del informe, copiada inline: este notebook no
        # ejecuta scripts del repo. Solo la edición en inglés: el sitio en
        # español bloquea las descargas directas (WAF), así que la edición en
        # inglés es la que se descarga (mismo informe, mismas cifras).
        _PDF_URL_EN = "https://annualreport.femsa.com/pdf/FEMSA_2024_Annual_Report.pdf"
        _PDF_PATH_EN = Path(DATA_DIR) / "FEMSA_2024_Annual_Report_EN.pdf"
        _SOURCE_LABEL_EN = "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)"

        def _inline_is_valid_pdf(path):
            try:
                return len(PdfReader(str(path)).pages) > 0
            except Exception:
                return False

        def _inline_download_pdf():
            if _PDF_PATH_EN.exists() and _inline_is_valid_pdf(_PDF_PATH_EN):
                print(f"PDF en caché: {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
                return _PDF_PATH_EN
            if _PDF_PATH_EN.exists():
                print(f"Caché inválido, se vuelve a descargar: {_PDF_PATH_EN}")
                _PDF_PATH_EN.unlink()
            Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Descargando {_PDF_URL_EN} ...")
            req = urllib.request.Request(
                _PDF_URL_EN, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(_PDF_PATH_EN, "wb") as fh:
                    fh.write(resp.read())
            if not _inline_is_valid_pdf(_PDF_PATH_EN):
                _PDF_PATH_EN.unlink()
                raise RuntimeError("El archivo descargado no es un PDF válido.")
            print(f"Guardado en {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
            return _PDF_PATH_EN

        def _inline_extract_pages(pdf_path):
            reader = PdfReader(str(pdf_path))
            chunks = []
            for i, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                chunks.append({"page": i, "text": text, "chars": len(text),
                               "fuente": _SOURCE_LABEL_EN})
            return chunks

        try:
            _pdf = _inline_download_pdf()
            _chunks = _inline_extract_pages(_pdf)
            _non_empty = sum(1 for c in _chunks if c["chars"] > 0)
            print(f"Páginas: {len(_chunks)} ({_non_empty} con texto)")
            Path(CHUNKS).parent.mkdir(parents=True, exist_ok=True)
            with open(CHUNKS, "w", encoding="utf-8") as fh:
                for c in _chunks:
                    fh.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"{len(_chunks)} páginas -> {CHUNKS}")
        except Exception as e:
            print("Generación inline falló:", e)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 rank-bm25==0.2.2 langchain-ollama

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", JUDGE_OLLAMA_MODEL)

## 1. Retrieval híbrido top-10 (inline)

Lógica inline (procedencia: `rag_baseline/retrieve_hybrid.py`): BM25 Okapi + MiniLM, fusión RRF k=60, top-10 por pregunta (el juez necesita más de donde elegir que el top-5 fijo). Determinista: se puede regenerar sin cambiar el resultado. Escribe `retrieval_hybrid_k10.json` con el mismo esquema que el script (página, score RRF, `bm25_rank`/`dense_rank` y scores por chunk, cobertura de cifras esperadas y top-10 de cada ranker).

In [ ]:
import json, re, time

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RRF_K = 60
EVAL_PATH = REPO + "/memo/eval_questions.json"
TOKEN_RE = re.compile(r"[a-z0-9áéíóúüñ]+")


def tokenize(text):
    return TOKEN_RE.findall((text or "").lower())


def digits(s):
    return re.sub(r"[.,]", "", s)


def numbers_in(text):
    return {digits(m.group(0)) for m in re.finditer(r"\d[\d.,]*", text or "")}


def rrf_ranks(bm25_scores, dense_sims):
    # Ranks 1-based (1 = mejor) de cada ranker sobre los 151 chunks.
    bm25_rank = np.empty(len(bm25_scores), dtype=int)
    bm25_rank[np.argsort(-np.asarray(bm25_scores))] = np.arange(1, len(bm25_scores) + 1)
    dense_rank = np.empty(len(dense_sims), dtype=int)
    dense_rank[np.argsort(-np.asarray(dense_sims))] = np.arange(1, len(dense_sims) + 1)
    return bm25_rank, dense_rank


t0 = time.time()
chunks = [json.loads(l) for l in open(CHUNKS)]
texts = [c["text"] for c in chunks]
print(f"BM25 Okapi (k1=1.5, b=0.75) + {EMBED_MODEL} sobre {len(chunks)} chunks ...",
      flush=True)
tokenized = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized)
print(f"BM25 index listo en {time.time() - t0:.1f}s", flush=True)

model = SentenceTransformer(EMBED_MODEL)
emb = np.asarray(model.encode(texts, normalize_embeddings=True,
                              show_progress_bar=False), dtype=np.float32)
print(f"embeddings: {emb.shape} en {time.time() - t0:.1f}s", flush=True)

eval_qs = json.load(open(EVAL_PATH))["questions"]
results = []
for q in eval_qs:
    q_tokens = tokenize(q["question"])
    b_scores = np.asarray(bm25.get_scores(q_tokens), dtype=np.float64)
    q_emb = model.encode([q["question"]], normalize_embeddings=True,
                         show_progress_bar=False)[0]
    sims = emb @ q_emb
    b_rank, d_rank = rrf_ranks(b_scores, sims)
    rrf = 1.0 / (RRF_K + b_rank) + 1.0 / (RRF_K + d_rank)
    top = np.argsort(-rrf)[:K]
    retrieved = []
    for idx in top:
        c = chunks[int(idx)]
        retrieved.append({"page": c["page"],
                          "score": round(float(rrf[idx]), 6),
                          "bm25_rank": int(b_rank[idx]),
                          "dense_rank": int(d_rank[idx]),
                          "bm25_score": round(float(b_scores[idx]), 4),
                          "dense_score": round(float(sims[idx]), 4),
                          "text": c["text"]})
    union_nums = set()
    for r in retrieved:
        union_nums |= numbers_in(r["text"])
    coverage = {}
    for e in q.get("expected_numbers", []):
        d = digits(str(e))
        hit = d in union_nums
        if not hit:
            try:
                v = float(e)
                hit = any(abs(float(a) - v) <= max(1.0, abs(v) * 0.001)
                          for a in union_nums if a.replace(".", "", 1).isdigit())
            except ValueError:
                hit = False
        coverage[e] = bool(hit)
    b_top10 = [int(chunks[i]["page"]) for i in np.argsort(-b_scores)[:10]]
    d_top10 = [int(chunks[i]["page"]) for i in np.argsort(-sims)[:10]]
    results.append({"id": q["id"], "level": q["level"],
                    "question": q["question"],
                    "expected": q["expected"],
                    "expected_numbers": q.get("expected_numbers", []),
                    "retrieved": retrieved,
                    "coverage": coverage,
                    "all_covered": all(coverage.values()) if coverage else None,
                    "bm25_top10_pages": b_top10,
                    "dense_top10_pages": d_top10})
    cov = "ALL" if results[-1]["all_covered"] else (
        "PARTIAL" if any(coverage.values()) else "NONE")
    print(f"{q['id']} L{q['level']} coverage={cov} "
          f"pages={[r['page'] for r in retrieved]} "
          f"(bm25_top10={b_top10[:5]} dense_top10={d_top10[:5]})", flush=True)

out = {"method": "hybrid BM25Okapi (k1=1.5, b=0.75) + MiniLM dense, RRF k=60",
       "embed_model": EMBED_MODEL, "k": K, "n_chunks": len(chunks),
       "retrieval_seconds": round(time.time() - t0, 1),
       "questions": results}
json.dump(out, open(RETRIEVAL_K10, "w"), ensure_ascii=False, indent=2)
n_all = sum(1 for r in results if r["all_covered"])
n_none = sum(1 for r in results if r["all_covered"] is False
             and not any(r["coverage"].values()))
print(f"retrieval hibrido: {n_all}/12 preguntas con TODAS las cifras "
      f"esperadas en top-{K}; {n_none}/12 sin ninguna -> {RETRIEVAL_K10}")

## 2. Juez + lector (ChatOllama)

Un llamado al juez por pregunta con los 10 chunks numerados (misma versión condensada que verá el lector; SYSTEM del juez, parseo de veredictos, SYSTEM del lector y `condense()` van inline en la celda de abajo, procedencia: `agentic_rag/judge_rerank.py` y `rag_baseline/answer.py`; máximo 3 reintentos; si no hay veredictos válidos, abstención sin lector). El lector contesta solo con los chunks seleccionados. Reanudable: salta las preguntas cuyo id ya está en la traza (`run_trace_judge_colab.jsonl`, nunca toca la traza del run en CPU).

In [ ]:
import json, re, time, unicodedata
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

ABSTAIN = "No lo sé."

# --- Juez: SYSTEM y parseo de veredictos (inline) ---
JUDGE_SYSTEM = (
    "Eres un juez que decide si un fragmento del informe anual de FEMSA "
    "2024 sirve para responder una pregunta. Se te presentan varios "
    "fragmentos numerados del 1 al 10. Devuelve UNA linea por fragmento, "
    "en este formato exacto: 'N: UTIL: <razon corta>' si el fragmento "
    "contiene informacion que ayuda a responder la pregunta, o "
    "'N: NO UTIL: <razon corta>' si no la contiene. Una linea por "
    "fragmento, sin texto adicional, sin resumen, sin numeracion extra.")

_LINE_RE = re.compile(r"^\s*(\d{1,2})\s*[:\.\)\-]\s*(.+)$", re.MULTILINE)


def _norm(s):
    # Minusculas sin acentos, para comparar 'UTIL'/'ÚTIL'/'util' igual.
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn").lower()


def _parse_line(rest):
    # Devuelve (useful, reason) o None si no se puede parsear.
    # Se revisa NO UTIL antes que UTIL porque 'NO UTIL' contiene 'UTIL'.
    rest = rest.strip()
    low = _norm(rest)
    if re.match(r"^(no\s+util|no\b)", low):
        reason = re.sub(r"^(no\s+[uú]til|no)\s*:?\s*", "", rest, flags=re.I)
        return (False, reason.strip()[:120])
    if re.match(r"^util\b", low):
        reason = re.sub(r"^[uú]til\s*:?\s*", "", rest, flags=re.I)
        return (True, reason.strip()[:120])
    if re.match(r"^si\b", low):
        reason = re.sub(r"^s[íi]\s*:?\s*", "", rest, flags=re.I)
        return (True, reason.strip()[:120])
    return None


def parse_verdicts(raw, n):
    # Devuelve {1: {"useful": bool, "reason": str}, ..., n: {...}} si hay
    # una linea parseable para CADA uno de los n chunks; None si falta
    # alguno. La primera linea valida por numero gana.
    verdicts = {}
    for m in _LINE_RE.finditer(raw or ""):
        idx = int(m.group(1))
        if idx < 1 or idx > n or idx in verdicts:
            continue
        parsed = _parse_line(m.group(2))
        if parsed is not None:
            verdicts[idx] = {"useful": parsed[0], "reason": parsed[1]}
    if set(verdicts.keys()) == set(range(1, n + 1)):
        return verdicts
    return None


# --- Lector: SYSTEM y condense (inline) ---
READER_SYSTEM = (
    "Responde la pregunta usando UNICAMENTE la información de los fragmentos "
    "del informe anual de FEMSA 2024 que se te dan a continuación. Copia las "
    "cifras exactamente como aparecen en los fragmentos, sin redondear ni "
    "inventar nada. Solo si la pregunta pide explícitamente sumar o comparar "
    "cifras que están en los fragmentos, haz esa operación con esas cifras. "
    "Si la información de los fragmentos no alcanza para responder, responde "
    "exactamente: No lo sé. Responde en español, en una o dos frases cortas.")

MAX_CHUNK_CHARS = 1800
STOPWORDS = set("de la el en y a los del se las por un para con no una su al lo como\nmas pero sus le ya o este mi que qué cuando donde cuales cuál cómo cuánto".split())


def condense(text, question, max_chars=MAX_CHUNK_CHARS):
    # ~max_chars caracteres por chunk: ventanas alrededor de las 2 lineas
    # del chunk mas relevantes a la pregunta (overlap de palabras + lineas
    # con cifras). Dos ventanas porque las dos cifras que una pregunta
    # compara suelen estar lejos en la misma pagina.
    qwords = {w.lower() for w in re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", question)
              if len(w) > 3 and w.lower() not in STOPWORDS}
    lines = text.splitlines() or [text]
    offsets, pos = [], 0
    for ln in lines:
        offsets.append(pos)
        pos += len(ln) + 1

    def line_score(ln):
        low = ln.lower()
        s = sum(2 for w in qwords if w in low)
        if re.search(r"\d", ln):
            s += 1
        return s

    scored = sorted(((line_score(ln), i) for i, ln in enumerate(lines)
                     if line_score(ln) > 0), reverse=True)
    chosen = []
    for s, i in scored:
        if all(abs(i - j) > 3 for j in chosen):
            chosen.append(i)
        if len(chosen) == 2:
            break
    if not chosen:
        return text[:max_chars]
    half = max_chars // max(len(chosen), 1)
    parts = []
    for i in sorted(chosen):
        start = offsets[i]
        lo = max(0, start - half // 3)
        hi = min(len(text), lo + half)
        lo = max(0, hi - half)
        parts.append(text[lo:hi])
    out = "\n[...]\n".join(parts)
    return out[:max_chars]


judge_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                       temperature=0, num_predict=200)
reader_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                        temperature=0, num_predict=100)
print("juez/lector (Ollama):", JUDGE_OLLAMA_MODEL)

def judge_once(question, chunks):
    # chunks: lista de (page, condensed_text). Un llamado, 10 veredictos.
    parts = [f"[{i}] Página {p}\n{c}" for i, (p, c) in enumerate(chunks, start=1)]
    user = (f"Pregunta: {question}\n\nFragmentos del informe anual "
            f"de FEMSA 2024:\n\n" + "\n\n".join(parts))
    return judge_llm.invoke([SystemMessage(content=JUDGE_SYSTEM),
                             HumanMessage(content=user)]).content.strip()

def run_question(item):
    t0 = time.time()
    chunks = [(r["page"], condense(r["text"], item["question"]))
              for r in item["retrieved"]]
    presented = [{"n": i + 1, "page": p, "condensed": c}
                 for i, (p, c) in enumerate(chunks)]
    rec = {"id": item["id"], "level": item["level"], "question": item["question"],
           "decomposition": "judge-rerank",
           "reader_model": f"ollama {JUDGE_OLLAMA_MODEL}",
           "retrieved_pages": [r["page"] for r in item["retrieved"]],
           "retrieval_scores": [r["score"] for r in item["retrieved"]],
           "chunks_presented": presented,
           "judge": {"attempts": 0, "attempts_raw": [], "verdicts": None,
                     "ok": False, "seconds": 0.0},
           "selected_pages": [],
           "reader": {"attempts": 0, "seconds": 0.0, "failed": False, "skipped": False},
           "context": "", "final_answer": ABSTAIN, "seconds": 0.0}
    # Juez: maximo 3 reintentos si no se pueden parsear los veredictos.
    tj = time.time()
    verdicts = None
    for attempt in range(1, 4):
        try:
            raw = judge_once(item["question"], chunks)
        except Exception as e:
            raw = f"<ERROR: {type(e).__name__}: {e}>"
        rec["judge"]["attempts_raw"].append(raw)
        rec["judge"]["attempts"] = attempt
        verdicts = parse_verdicts(raw, len(chunks))
        if verdicts is not None:
            break
    rec["judge"]["seconds"] = round(time.time() - tj, 1)
    if verdicts is None:
        rec["judge"]["ok"] = False
        rec["reader"]["skipped"] = True
        rec["seconds"] = round(time.time() - t0, 1)
        return rec
    rec["judge"]["ok"] = True
    rec["judge"]["verdicts"] = verdicts
    selected = [i for i in range(1, len(chunks) + 1) if verdicts[i]["useful"]]
    rec["selected_pages"] = [chunks[i - 1][0] for i in selected]
    if not selected:
        rec["reader"]["skipped"] = True
        rec["seconds"] = round(time.time() - t0, 1)
        return rec
    ctx = "\n\n".join(f"[Página {chunks[i - 1][0]}]\n{chunks[i - 1][1]}" for i in selected)
    rec["context"] = ctx
    # Lector: maximo 3 reintentos si el llamado falla o viene vacio.
    tr = time.time()
    answer, failed = None, False
    user = (f"Fragmentos del informe anual de FEMSA 2024:\n\n{ctx}\n\n"
            f"Pregunta: {item['question']}")
    for attempt in range(1, 4):
        try:
            ans = reader_llm.invoke([SystemMessage(content=READER_SYSTEM),
                                     HumanMessage(content=user)]).content.strip()
        except Exception as e:
            ans = ""
            print(f"  {item['id']} lector intento {attempt} fallo: {type(e).__name__}: {e}", flush=True)
        if ans:
            answer = ans
            rec["reader"]["attempts"] = attempt
            break
        rec["reader"]["attempts"] = attempt
    else:
        failed = True
    rec["reader"]["seconds"] = round(time.time() - tr, 1)
    rec["reader"]["failed"] = failed
    rec["final_answer"] = ABSTAIN if failed else answer
    rec["seconds"] = round(time.time() - t0, 1)
    return rec

done = set()
try:
    for line in open(TRACE_OUT):
        try:
            done.add(json.loads(line)["id"])
        except (json.JSONDecodeError, KeyError):
            pass
except FileNotFoundError:
    pass
if done:
    print(f"reanudando: {sorted(done)} ya en {TRACE_OUT}")

qs = json.load(open(RETRIEVAL_K10))["questions"]
t0 = time.time()
with open(TRACE_OUT, "a") as f:
    for item in qs:
        if item["id"] in done:
            print(f"{item['id']} ya hecha, salto", flush=True)
            continue
        print(f"{item['id']} L{item['level']}: {item['question']}", flush=True)
        rec = run_question(item)
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        print(f"{item['id']} juez_ok={rec['judge']['ok']} "
              f"intentos_juez={rec['judge']['attempts']} "
              f"seleccionados={len(rec['selected_pages'])}/10 "
              f"lector_intentos={rec['reader']['attempts']} "
              f"-> {rec['final_answer'][:160]}", flush=True)
print(f"listo en {time.time() - t0:.1f}s -> {TRACE_OUT}")

## 3. Puntuar con el mismo scorer (con diagnóstico del juez, inline)

Lógica inline (procedencia: `agentic_rag/score_judge.py`): escribe `results_agentic_judge_colab.md` con el puntaje, la comparación con los demás sistemas y el diagnóstico del juez (precisión/recall de selección contra las páginas de evidencia). Misma exactitud numérica, misma definición de alucinación, misma regla Q3 con 1,596 y mismas keywords Q8/Q10/Q11 que los demás scorers. Los modos de fallo quedan en PENDIENTE: se verifican a mano contra el corpus y se llenan en `VERIFIED_MODES` de la celda de abajo, y se vuelve a correr la celda con `BLANK_MODES = False`.

In [ ]:
import json, re
from collections import Counter

TRACE_PATH = TRACE_OUT
OUT_MD = REPO + "/agentic_rag/results_agentic_judge_colab.md"
EVAL_PATH = REPO + "/memo/eval_questions.json"
QA_PATH = REPO + "/memo/reflection_qa.jsonl"
SEED_PATH = REPO + "/pipeline/triples_seed.json"
GT_PATH = REPO + "/rag_baseline/ragas/ground_truth_evidence.json"
MODEL_LABEL = f"Ollama {JUDGE_OLLAMA_MODEL} (Colab T4)"
SYSTEM_NAME = "RAG agentico juez Ollama (Colab)"
# BLANK_MODES = True: modos de fallo en PENDIENTE (run nuevo). Tras
# verificarlos a mano, llena VERIFIED_MODES abajo y corre con False.
BLANK_MODES = True
VERIFIED_MODES = {f"Q{i}": "PENDIENTE" for i in range(1, 13)}
if BLANK_MODES:
    VERIFIED_MODES = {qid: "PENDIENTE" for qid in VERIFIED_MODES}


def digits(s):
    return re.sub(r"[.,]", "", s)


def numbers_in(text):
    return {digits(m.group(0)) for m in re.finditer(r"\d[\d.,]*", text or "")}


def norm_num(x):
    return digits(str(x))


def expected_hit(answer_nums, expected):
    # True si cada numero esperado aparece (o su redondeo a entero).
    for e in expected:
        d = norm_num(e)
        if d in answer_nums:
            continue
        try:
            v = float(e)
            ok = any(abs(float(a) - v) <= max(1.0, abs(v) * 0.001)
                     for a in answer_nums if a.replace(".", "", 1).isdigit())
        except ValueError:
            ok = False
        if not ok:
            return False
    return True


def literal_present(text, expected):
    # True si la cifra esperada aparece literalmente en el texto (con ','
    # o '.' como separador). Sin tolerancia de redondeo.
    e = str(expected).replace(",", "")
    if "." in e:
        return e in text or e.replace(".", ",") in text
    grouped = f"{int(e):,}"
    return grouped in text or e in text


eval_data = json.load(open(EVAL_PATH))
questions = {q["id"]: q for q in eval_data["questions"]}
trace = [json.loads(l) for l in open(TRACE_PATH)]

# Mismo conjunto de numeros verificados que el scorer de MEMO.
verified = set()
try:
    for line in open(QA_PATH):
        verified |= numbers_in(json.loads(line)["answer"])
except FileNotFoundError:
    pass
try:
    seed = json.load(open(SEED_PATH))
    triples = seed if isinstance(seed, list) else seed.get("triples", [])
    for t in triples:
        for k in ("valor", "variacion_pct"):
            if t.get(k) is not None:
                verified.add(digits(str(t[k])))
except FileNotFoundError:
    pass
verified |= {"2024", "2023", "2022"}

# Paginas de evidencia por pregunta (ground truth auditable, RAGAS).
evidence_pages = {}
try:
    gt = json.load(open(GT_PATH))
    for q in gt.get("questions", []):
        pages = set()
        for item in q.get("evidence", []):
            pages |= set(item.get("pages", []))
        evidence_pages[q["id"]] = pages
except FileNotFoundError:
    evidence_pages = {}

rows = []
for r in trace:
    q = questions[r["id"]]
    ans = r["final_answer"]
    ans_nums = numbers_in(ans)
    halluc = sorted(n for n in ans_nums if n not in verified and len(n) > 1)
    ok = expected_hit(ans_nums, q.get("expected_numbers", []))
    kws = q.get("expected_keywords", [])
    if not q.get("expected_numbers") and kws:
        ok = all(k.lower() in ans.lower() for k in kws)
    elif not q.get("expected_numbers") and not kws:
        ok = False
    note = ""
    if q.get("ground_truth_note"):
        note = "Nota ground-truth: " + q["ground_truth_note"]
    if r["id"] == "Q3" and "1596" in ans_nums:
        ok = True
        note += " (El modelo respondio 1,596 = valor real del informe; se cuenta como correcto.)"
    verdict = "correcta" if ok and not halluc else (
        "correcta con alucinacion" if ok else "incorrecta")
    exp_nums = q.get("expected_numbers", [])
    presented_text = "\n".join(c.get("condensed", "")
                               for c in r.get("chunks_presented", []))
    cov_presented = {e: literal_present(presented_text, e) for e in exp_nums}
    cov_selected = {e: literal_present(r.get("context", ""), e) for e in exp_nums}
    ev = evidence_pages.get(r["id"], set())
    sel = set(r.get("selected_pages", []))
    sel_prec = (len(sel & ev) / len(sel)) if sel else None
    sel_rec = (len(sel & ev) / len(ev)) if ev else None
    judge = r.get("judge", {})
    mode = VERIFIED_MODES[r["id"]]
    if verdict == "correcta":
        mode = "ok"
    rows.append({"id": r["id"], "level": q["level"],
                 "question": q["question"], "expected": q["expected"],
                 "answer": ans, "verdict": verdict,
                 "hallucinated": halluc, "note": note,
                 "judge_ok": judge.get("ok"),
                 "judge_attempts": judge.get("attempts"),
                 "judge_seconds": judge.get("seconds", 0),
                 "n_selected": len(sel),
                 "selected_pages": sorted(sel),
                 "sel_precision": sel_prec,
                 "sel_recall": sel_rec,
                 "evidence_pages": sorted(ev),
                 "reader_attempts": r.get("reader", {}).get("attempts", 0),
                 "reader_skipped": r.get("reader", {}).get("skipped", False),
                 "reader_failed": r.get("reader", {}).get("failed", False),
                 "seconds": r.get("seconds", 0),
                 "coverage_presented": cov_presented,
                 "coverage_selected": cov_selected,
                 "failure_mode": mode})

by_level = {}
for row in rows:
    by_level.setdefault(row["level"], []).append(row)
total_ok = sum(1 for r in rows if r["verdict"] == "correcta")
total_halluc = sum(1 for r in rows if r["hallucinated"])
n_ev_full = sum(1 for r in rows if r["coverage_selected"]
                and all(r["coverage_selected"].values()))
n_judge_ok = sum(1 for r in rows if r["judge_ok"])
n_abstain_judge = sum(1 for r in rows if not r["judge_ok"])
n_abstain_empty = sum(1 for r in rows
                      if r["judge_ok"] and r["n_selected"] == 0)
total_selected = sum(r["n_selected"] for r in rows)
sel_inter = sum(len(set(r["selected_pages"]) & set(r["evidence_pages"]))
                for r in rows)
sel_total = sum(len(r["selected_pages"]) for r in rows)
ev_total = sum(len(r["evidence_pages"]) for r in rows)
pooled_prec = (sel_inter / sel_total) if sel_total else None
pooled_rec = (sel_inter / ev_total) if ev_total else None
total_judge_seconds = sum(r["judge_seconds"] for r in rows)
total_seconds = sum(r["seconds"] for r in rows)
pending = [r["id"] for r in rows if r["failure_mode"] == "PENDIENTE"]
if pending:
    print(f"AVISO: modos de fallo PENDIENTES de verificar a mano: {pending}")

def lvl_ok(lv):
    return sum(1 for r in by_level.get(lv, [])
               if r["verdict"] == "correcta")

def pct(x):
    return f"{x:.2f}" if x is not None else "n/a"

lines = ["# Resultados RAG agentico juez (seleccion de chunks con juez) — FEMSA 2024",
         "",
         "Sistema: retrieval hibrido top-10 (BM25 Okapi + MiniLM, RRF k=60), "
         f"UN llamado al juez por pregunta que califica los 10 chunks de golpe "
         f"(UTIL / NO UTIL, modelo {MODEL_LABEL}), maximo 3 reintentos si los "
         "veredictos no se pueden parsear (si a los 3 sigue sin veredictos validos, "
         "la pregunta se abstiene sin llamar al lector), y el lector responde "
         "UNICAMENTE con los chunks seleccionados (mismo lector, mismo condense, "
         "mismo prompt del lector del RAG baseline; si el juez no selecciona nada, "
         "abstencion sin llamar al lector). Sin graph_query, sin calculator. "
         "Scorer con la misma exactitud numerica, misma definicion de "
         "alucinacion y misma regla Q3 con 1,596 que los demas sistemas.",
         "",
         f"Correctas: **{total_ok}/12**. Con alucinacion (numero no verificado en el informe): **{total_halluc}/12**. "
         f"Preguntas donde el contexto seleccionado contenia todas las cifras "
         f"esperadas (literal): **{n_ev_full}/12**. "
         f"Juez con veredictos validos: **{n_judge_ok}/12** preguntas "
         f"({n_abstain_judge} abstencion{'es' if n_abstain_judge != 1 else ''} por juez fallido, "
         f"{n_abstain_empty} por seleccion vacia). "
         f"Chunks seleccionados: **{total_selected}/120** "
         f"(media {total_selected / max(len(rows), 1):.1f} por pregunta). "
         f"Seleccion del juez vs evidencia (pooled): precision "
         f"**{pct(pooled_prec)}**, recall **{pct(pooled_rec)}**. "
         f"Tiempo total de la corrida: **{total_seconds / 60:.1f} min** "
         f"(juez {total_judge_seconds / 60:.1f} min, media "
         f"{total_seconds / max(len(rows), 1):.0f} s por pregunta).",
         "",
         "| ID | Nivel | Pregunta | Esperado | Respuesta del agente | Veredicto | Sel. | Juez | Modo de fallo |",
         "|----|-------|----------|----------|----------------------|-----------|------|------|---------------|"]
for r in rows:
    a = r["answer"].replace("|", "/").replace("\n", " ")[:200]
    judge_s = f"{r['judge_attempts']} int." if r["judge_ok"] else "fallo"
    lines.append(f"| {r['id']} | {r['level']} | {r['question'][:60]} | "
                 f"{r['expected'][:55]} | {a} | {r['verdict']} | "
                 f"{r['n_selected']}/10 | {judge_s} | {r['failure_mode']} |")
lines += ["",
          "## Por nivel",
          ""]
for lv in sorted(by_level):
    lines.append(f"- Nivel {lv}: {lvl_ok(lv)}/{len(by_level[lv])} correctas.")
lines += ["",
          "## Diagnostico del juez (seleccion vs evidencia)",
          "",
          "Precision = paginas seleccionadas que son evidencia / paginas "
          "seleccionadas. Recall = paginas de evidencia conservadas / paginas de "
          "evidencia totales (ground truth auditable, "
          "rag_baseline/ragas/ground_truth_evidence.json). Un recall bajo dice que "
          "el juez tiro la evidencia; una precision baja dice que el juez dejo pasar "
          "ruido y el lector tuvo que elegir entre mas chunks.",
          "",
          "| ID | Seleccionados | Precision | Recall | Paginas de evidencia |",
          "|----|---------------|-----------|--------|----------------------|"]
for r in rows:
    lines.append(f"| {r['id']} | {r['n_selected']}/10 ({', '.join(str(p) for p in r['selected_pages']) or 'ninguna'}) | "
                 f"{pct(r['sel_precision'])} | {pct(r['sel_recall'])} | "
                 f"{', '.join(str(p) for p in r['evidence_pages']) or 'n/a'} |")
lines += ["",
          f"**Pooled:** precision {pct(pooled_prec)} ({sel_inter}/{sel_total}), "
          f"recall {pct(pooled_rec)} ({sel_inter}/{ev_total}).",
          "",
          "## Modos de fallo",
          ""]
for mode, n in Counter(r["failure_mode"] for r in rows).most_common():
    ids = ", ".join(r["id"] for r in rows if r["failure_mode"] == mode)
    lines.append(f"- {mode}: {n} ({ids})")
lines += ["",
          "## Comparacion de los sistemas (12 preguntas, mismo scorer)",
          "",
          "| Sistema | Correctas | Nivel 1 | Nivel 2 | Nivel 3 |",
          "|---------|-----------|---------|---------|---------|",
          "| Grafo (Relate, Cypher) | 12/12 | 4/4 | 4/4 | 4/4 |",
          f"| {SYSTEM_NAME} | {total_ok}/12 | {lvl_ok(1)}/4 | {lvl_ok(2)}/4 | {lvl_ok(3)}/4 |",
          "| RAG hibrido + expansion de queries | 5/12 | 2/4 | 2/4 | 1/4 |",
          "| RAG hibrido BM25+denso (sin expansion) | 2/12 | 1/4 | 0/4 | 1/4 |",
          "| RAG vectorial denso puro | 2/12 | 0/4 | 1/4 | 1/4 |",
          "| RAG agentico ReAct (LangChain + LangGraph, 1.5B) | 2/12 | 1/4 | 1/4 | 0/4 |",
          "| MEMO run 2 (0.5B LoRA r=32 + ejecutivo 1.5B) | 2/12 | 1/4 | 1/4 | 0/4 |",
          "| MEMO run 1 (0.5B LoRA r=8 + ejecutivo 0.5B) | 0/12 | 0/4 | 0/4 | 0/4 |",
          ""]
if not BLANK_MODES:
    lines += ["## Que cambio vs el RAG hibrido + expansion (5/12)",
              "",
              "PENDIENTE de redactar tras verificar los modos de fallo a mano: "
              "comparar pregunta por pregunta cuales flips son por la seleccion del "
              "juez (tiro la evidencia / dejo pasar ruido) y cuales por el lector.",
              "",
              "## Veredicto para el post",
              "",
              "PENDIENTE de redactar tras verificar los modos de fallo a mano: la "
              "pregunta es si la seleccion con juez supera el top-5 fijo del pipeline "
              "hibrido+expansion (5/12) con el mismo modelo y el mismo scorer.",
              "",
              ]
else:
    lines += ["## Veredicto para el post",
              "",
              "Pendiente de revision manual. Los numeros de la tabla de arriba ya son "
              "comparables con los demas sistemas (mismo scorer, mismas 12 preguntas); "
              "lo que falta es verificar cada fallo contra el corpus y contra la "
              "traza, llenar VERIFIED_MODES en la celda de arriba y volver a correr "
              "el scorer con BLANK_MODES = False para que escriba los modos de fallo "
              "y el veredicto.",
              "",
              ]
lines += ["## Caveats (verificados, no supuestos)",
          "",
          "- **Q7 y Q9 son techo (9/12), igual que en los otros runs.** Las cifras "
          "esperadas de Q7 (+27.5%/+24.9%) vienen del comunicado 4Q24 y las de Q9 "
          "(39.3/35.8/75.1%) son participaciones calculadas: no estan en el informe. "
          "Q2 tambien pide +24.9% como parte de lo esperado. El techo real del juez "
          "sobre este set es 9/12, no 12/12.",
          "- **El juez ve la version condensada de cada chunk, la misma que vera el "
          "lector.** Un chunk es util si lo que el lector va a ver de el sirve para "
          "responder; la seleccion se mide contra esa misma version, no contra la "
          "pagina completa.",
          "- **El resto de los caveats esta pendiente.** Los modos de fallo de este "
          "run todavia no se verificaron a mano: ver la seccion de veredicto arriba.",
          "",
          "## Notas de ground-truth",
          ""]
for r in rows:
    if r["note"]:
        lines.append(f"- {r['id']}: {r['note']}")
if not any(r["note"] for r in rows):
    lines.append("- Ninguna.")
lines += ["",
          "## Alucinaciones detectadas",
          ""]
for r in rows:
    if r["hallucinated"]:
        lines.append(f"- {r['id']}: numeros no verificados en el informe: "
                     f"{', '.join(r['hallucinated'])}")
if not any(r["hallucinated"] for r in rows):
    lines.append("- Ninguna.")
lines += ["",
          "## Detalle de evidencia por pregunta (cobertura literal)",
          "",
          "| ID | Cifras esperadas en top-10 presentado | Cifras esperadas en contexto seleccionado |",
          "|----|----------------------------------------|-------------------------------------------|"]
for r in rows:
    cov_p = ", ".join(f"{k}: {'si' if v else 'no'}"
                      for k, v in r["coverage_presented"].items()) \
        if r["coverage_presented"] else "(sin cifras esperadas)"
    cov_s = ", ".join(f"{k}: {'si' if v else 'no'}"
                      for k, v in r["coverage_selected"].items()) \
        if r["coverage_selected"] else "(sin cifras esperadas)"
    lines.append(f"| {r['id']} | {cov_p} | {cov_s} |")

open(OUT_MD, "w").write("\n".join(lines) + "\n")
print(f"correctas {total_ok}/12, con alucinacion {total_halluc}/12, "
      f"seleccionados {total_selected}/120, juez ok {n_judge_ok}/12, "
      f"precision {pct(pooled_prec)}, recall {pct(pooled_rec)}, "
      f"{total_seconds / 60:.1f} min -> {OUT_MD}")
for r in rows:
    print(f"  {r['id']} L{r['level']} {r['verdict']} [{r['failure_mode']}] "
          f"sel={r['n_selected']}/10 juez={'ok' if r['judge_ok'] else 'fallo'} "
          f"({r['judge_attempts']} int.) {r['seconds']:.0f}s"
          + (f" halluc={r['hallucinated']}" if r["hallucinated"] else ""))

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El puntaje del juez es EL resultado agéntico que va al post (el ReAct 2/12 queda como negativo documentado). Sin esta traza, el draft no se puede actualizar con el resultado real.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('rag_baseline/retrieval_hybrid_k10.json', 'Retrieval híbrido top-10 (12 preguntas)', True), ('agentic_rag/run_trace_judge_colab.jsonl', 'Traza juez + lector (12 preguntas)', True), ('agentic_rag/results_agentic_judge_colab.md', 'Resultados RAG agéntico juez', True)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)